# Chapter 2 — Vectors: The Geometry Behind Language Models

**Central question:**

> What mathematical object carries information through a neural language model?

Chapter 1 exposed you to rows of numbers inside tensors and embedding tables. This chapter slows down and explains what those rows actually are, how they can be compared and transformed, and why vector operations become foundational to learned representations, attention, retrieval, and neural computation.

Every concept introduced here is used later in the book.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import string
import urllib.request

# ── Rebuild exactly from 01a_data_pipeline_amar.ipynb ──────────────────────
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
urllib.request.urlretrieve(url, "tinyshakespeare.txt")

with open("tinyshakespeare.txt", "r", encoding="utf-8") as f:
    txt = f.read().replace('\n', '')

# 95-character vocabulary (same as 01a_data_pipeline_amar)
chars      = sorted(set(txt) | set(string.ascii_letters) | set(string.digits) | set(string.punctuation))
vocab_size = len(chars)
stoi       = {ch: i for i, ch in enumerate(chars)}
itos       = {i: ch for i, ch in enumerate(chars)}
encode     = lambda s: [stoi[c] for c in s]
decode     = lambda l: ''.join([itos[i] for i in l])
data       = torch.tensor(encode(txt), dtype=torch.long)

device = 'cuda' if torch.cuda.is_available() else 'cpu'

n          = int(0.9 * len(data))
train_data = data[:n]
val_data   = data[n:]

block_size = 8
batch_size = 32

def get_batch(split='train'):
    d  = train_data if split == 'train' else val_data
    ix = torch.randint(len(d) - block_size, (batch_size,))
    x  = torch.stack([d[i:i+block_size]     for i in ix])
    y  = torch.stack([d[i+1:i+block_size+1] for i in ix])
    return x.to(device), y.to(device)

print(f"Vocabulary size : {vocab_size}")
print(f"Total tokens    : {len(data):,}")
print(f"Train tokens    : {len(train_data):,}")
print(f"Device          : {device}")
print(f"encode('hello') = {encode('hello')}")

## 1. From Chapter 1's Numbers to Vectors

In Chapter 1 the bigram model used:

```python
nn.Embedding(vocab_size, vocab_size)
```

One lookup returned `vocab_size` numbers — used directly as logits.

Chapter 2 asks a more basic question:

> What exactly is a list of numbers?

The answer: a **vector** — an ordered collection of numbers with a fixed dimension.

In [ ]:
# A vector is an ordered collection of numbers with a fixed dimension
v = torch.tensor([0.2, -1.1, 0.7, 0.4])
print(f'v = {v}')
print(f'dimension: {v.shape[0]}')
print(f'dtype:     {v.dtype}')

# Any row of a weight matrix is also a vector
# The Chapter 1 model had a (95, 95) weight matrix -- same shape shown here
W_example = torch.randn(vocab_size, vocab_size)
one_row = W_example[0]
print(
    f"\nOne row of a ({vocab_size}, {vocab_size}) matrix: "
    f"shape={one_row.shape} "
    f"(a {vocab_size}-dimensional vector)"
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('A vector — ordered list of numbers', fontsize=12, fontweight='bold')

# ── Panel 1: vector as a row of numbers (memory layout) ──────────────────────
ax = axes[0]
ax.axis('off'); ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.set_title('Memory layout — ordered, indexed', fontsize=10)

vals = [0.2, -1.1, 0.7, 0.4]
cw, ch = 0.18, 0.22
sx = 0.5 - len(vals) * (cw + 0.03) / 2
for i, v in enumerate(vals):
    x = sx + i * (cw + 0.03)
    fc = '#d6eaf8' if v >= 0 else '#fadbd8'
    ax.add_patch(plt.Rectangle((x, 0.44), cw, ch, fc=fc, ec='#1a5276', lw=1.5))
    ax.text(x + cw/2, 0.55, f'{v}', ha='center', va='center',
            fontsize=11, color='#1c2833', fontweight='bold')
    ax.text(x + cw/2, 0.39, f'[{i}]', ha='center', fontsize=8.5, color='#7f8c8d')

ax.text(0.5, 0.80, 'v = [0.2, −1.1, 0.7, 0.4]',
        ha='center', fontsize=11, fontweight='bold', color='#1a5276')
ax.text(0.5, 0.28, 'shape: (4,)   ndim: 1   numel: 4',
        ha='center', fontsize=9, color='#555')
ax.text(0.5, 0.14, 'dtype: float32',
        ha='center', fontsize=9, color='#117a65')

# ── Panel 2: embedding table row = vector ────────────────────────────────────
ax2 = axes[1]
ax2.axis('off'); ax2.set_xlim(0, 6); ax2.set_ylim(0, 4)
ax2.set_title('Chapter 1 connection — embedding table row is a vector', fontsize=10)

V_show, C_show = 5, 6
cw2, ch2, gap = 0.72, 0.45, 0.06
sx2, sy2 = 0.35, 1.2
row_colors = ['#d6eaf8', '#d5f5e3', '#fdebd0', '#e8daef', '#fadbd8']
row_labels = ["'a'", "'b'", "' '", "'!'", '...']

for i, (rc, rl) in enumerate(zip(row_colors, row_labels)):
    y = sy2 + (V_show - 1 - i) * (ch2 + gap)
    for j in range(C_show):
        x = sx2 + j * (cw2 + gap)
        alpha = 1.0 if i == 2 else 0.4
        ax2.add_patch(plt.Rectangle((x, y), cw2, ch2,
                                    fc=rc, ec='#7f8c8d', lw=1.0, alpha=alpha))
    ax2.text(sx2 - 0.08, y + ch2/2, rl, ha='right', va='center',
             fontsize=9, color='#1c2833',
             fontweight='bold' if i == 2 else 'normal',
             alpha=1.0 if i == 2 else 0.45)

# highlight arrow on row 2
hi_y = sy2 + 2 * (ch2 + gap)
ax2.annotate('', xy=(sx2 + C_show*(cw2+gap) - gap + 0.08, hi_y + ch2/2),
             xytext=(sx2 + C_show*(cw2+gap) - gap + 0.55, hi_y + ch2/2),
             arrowprops=dict(arrowstyle='<-', color='#c0392b', lw=2.0))
ax2.text(sx2 + C_show*(cw2+gap) - gap + 0.65, hi_y + ch2/2,
         "row for ' '\n= 1 vector\nshape: (95,)",
         ha='left', va='center', fontsize=8.5, color='#c0392b', fontweight='bold')

ax2.text(sx2 + C_show*(cw2+gap)/2, 0.65,
         'E ∈ ℝ^(V × C)     lookup = select one row',
         ha='center', fontsize=9.5, color='#1c2833',
         bbox=dict(boxstyle='round,pad=0.3', fc='#f8f9fa', ec='#aab7b8', lw=1.0))

plt.tight_layout()
plt.show()

## 2. Scalar, Vector, Matrix, Tensor

All four objects already appeared in Chapter 1. Here we name them explicitly.

| Object | Dimensions | Example use |
|--------|-----------|-------------|
| Scalar | 0-D (one number) | loss value, learning rate |
| Vector | 1-D | token representation, logits |
| Matrix | 2-D | lookup table, weight matrix |
| Tensor | N-D | batch of sequences `(B, T, C)` |

In [ ]:
# Scalar — one number
loss = torch.tensor(2.51)
print(f"scalar: {loss}  shape: {loss.shape}")

# Vector — 1-D
logits = torch.tensor([0.1, 2.8, -0.3, 0.4])
print(f"vector: {logits}  shape: {logits.shape}")

# Matrix — 2-D
W = torch.randn(4, 8)
print(f"matrix: shape={W.shape}  ({W.shape[0]} rows × {W.shape[1]} cols)")

# Tensor — 3-D  (B=2, T=4, C=8)
x = torch.randn(2, 4, 8)
print(f"tensor: shape={x.shape}  (batch × sequence × features)")

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(14, 4))
fig.suptitle('The four tensor types', fontsize=13, fontweight='bold', y=1.02)

colors = {'box': '#d6eaf8', 'edge': '#1a5276', 'text': '#1c2833'}

# ── Panel 1: Scalar (rank 0) ─────────────────────────────────────────────────
ax = axes[0]
ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis('off')
ax.add_patch(plt.Circle((0.5, 0.55), 0.22, color='#f9e79f', ec='#b7950b', lw=2))
ax.text(0.5, 0.55, '2.51', ha='center', va='center', fontsize=16, fontweight='bold', color='#7d6608')
ax.text(0.5, 0.18, 'Scalar\nrank 0  |  shape: ()', ha='center', fontsize=10, color=colors['text'])
ax.text(0.5, 0.92, 'one number', ha='center', fontsize=9, color='gray', style='italic')

# ── Panel 2: Vector (rank 1) ─────────────────────────────────────────────────
ax = axes[1]
ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis('off')
vals = [0.1, 2.8, -0.3, 0.4]
cell_w = 0.18; gap = 0.03; start_x = 0.5 - (len(vals) * (cell_w + gap)) / 2
for i, v in enumerate(vals):
    x = start_x + i * (cell_w + gap)
    ax.add_patch(plt.Rectangle((x, 0.42), cell_w, 0.22,
                               color=colors['box'], ec=colors['edge'], lw=1.5))
    ax.text(x + cell_w/2, 0.53, f'{v}', ha='center', va='center', fontsize=8.5, color=colors['text'])
ax.annotate('', xy=(start_x + len(vals)*(cell_w+gap) - gap + 0.02, 0.53),
            xytext=(start_x - 0.02, 0.53),
            arrowprops=dict(arrowstyle='<->', color='#922b21', lw=1.5))
ax.text(0.5, 0.35, f'{len(vals)} elements', ha='center', fontsize=8.5, color='#922b21')
ax.text(0.5, 0.18, f'Vector\nrank 1  |  shape: ({len(vals)},)', ha='center', fontsize=10, color=colors['text'])
ax.text(0.5, 0.92, 'ordered list of numbers', ha='center', fontsize=9, color='gray', style='italic')

# ── Panel 3: Matrix (rank 2) ─────────────────────────────────────────────────
ax = axes[2]
ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis('off')
rows, cols = 3, 4; cw = 0.16; ch = 0.14; gap = 0.02
sx = 0.5 - (cols*(cw+gap))/2; sy = 0.36
for r in range(rows):
    for c in range(cols):
        ax.add_patch(plt.Rectangle((sx + c*(cw+gap), sy + r*(ch+gap)), cw, ch,
                                   color=colors['box'], ec=colors['edge'], lw=1.2))
ax.text(0.5, 0.18, f'Matrix\nrank 2  |  shape: ({rows}, {cols})', ha='center', fontsize=10, color=colors['text'])
ax.text(0.5, 0.92, 'rows × columns', ha='center', fontsize=9, color='gray', style='italic')

# ── Panel 4: Tensor (rank 3) ─────────────────────────────────────────────────
ax = axes[3]
ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis('off')
slabs = 3; sw = 0.38; sh = 0.30; dx = 0.06; dy = 0.04
bx = 0.28; by = 0.32
for s in reversed(range(slabs)):
    ox = bx + s*dx; oy = by + s*dy
    alpha = 0.55 + s * 0.15
    ax.add_patch(plt.Rectangle((ox, oy), sw, sh,
                               color='#aed6f1', ec=colors['edge'], lw=1.2, alpha=alpha))
    for row in range(1, 3):
        ax.plot([ox, ox+sw], [oy + row*sh/3]*2, color='#5d8aa8', lw=0.5, alpha=0.5)
    for col in range(1, 3):
        ax.plot([ox + col*sw/3]*2, [oy, oy+sh], color='#5d8aa8', lw=0.5, alpha=0.5)
ax.text(0.5, 0.18, 'Tensor (3-D)\nrank 3  |  shape: (B, T, C)', ha='center', fontsize=10, color=colors['text'])
ax.text(0.5, 0.92, 'generalises to any rank', ha='center', fontsize=9, color='gray', style='italic')

plt.tight_layout()
plt.show()

## 3. Rank, Shape, Number of Elements, and Dtype

These four properties are easy to mix up. Separate them early.

**Rank** — number of axes (`.ndim`):
```
scalar → rank 0
vector → rank 1
matrix → rank 2
(B,T,C) tensor → rank 3
```

**Shape** — size of each axis (`.shape`): tells you how many entries sit along each axis.

**Number of elements** — total stored values (`.numel()`): the product of all shape dimensions.

**Dtype** — the numerical type of the stored values:
```
token IDs          → integers  (torch.int64 / torch.long)
model calculations → floating-point  (torch.float32)
```

Shape tells you *how values are arranged*. Dtype tells you *what kind of values are stored*.

> Do not worry yet about `float16`, `bfloat16`, quantization, or numerical precision. Those belong later in the book (Chapter 16).
> **Tensor rank vs matrix rank:** This chapter uses *rank* to mean the number of axes (`.ndim`). Linear algebra uses the same word differently — the rank of a matrix is the number of linearly independent row or column directions. That is a separate concept introduced explicitly when it matters (Chapter 15, LoRA).


In [ ]:
a = torch.tensor(3.0)
b = torch.tensor([1.0, 2.0, 3.0])
c = torch.randn(4, 8)
d = torch.randn(4, 8, 16)

# Rank (number of axes)
print("--- Rank (.ndim) ---")
print(f"scalar: {a.ndim}")   # 0
print(f"vector: {b.ndim}")   # 1
print(f"matrix: {c.ndim}")   # 2
print(f"tensor: {d.ndim}")   # 3

# Shape (size of each axis)
print("\n--- Shape (.shape) ---")
print(f"vector: {b.shape}")  # (3,)
print(f"matrix: {c.shape}")  # (4, 8)
print(f"tensor: {d.shape}")  # (4, 8, 16)

# Number of elements
print("\n--- Elements (.numel()) ---")
print(f"tensor: {d.numel()}  (= 4 × 8 × 16)")

# Dtype
print("\n--- Dtype (.dtype) ---")
ids    = torch.tensor([1, 2, 3])
values = torch.tensor([1.0, 2.0, 3.0])
print(f"token IDs:    {ids.dtype}")     # torch.int64
print(f"float values: {values.dtype}")  # torch.float32
print(f"\nSame shape {ids.shape}, different dtypes — rank/shape and dtype are separate properties.")

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5.2))
ax.axis('off')
ax.set_title('Four tensor properties at a glance', fontsize=13, fontweight='bold', pad=12)

headers = ['Tensor', 'Rank (.ndim)', 'Shape (.shape)', 'numel()', 'dtype']
rows = [
    ['scalar  3.0',              '0', '()',          '1',   'float32  ← decimal literal'],
    ['vector  [1.0, 2.0, 3.0]', '1', '(3,)',        '3',   'float32  ← decimal literals'],
    ['vector  [1, 2, 3]',       '1', '(3,)',        '3',   'int64    ← integer literals'],
    ['matrix  randn(4, 8)',      '2', '(4, 8)',      '32',  'float32  ← randn always float'],
    ['tensor  randn(4, 8, 16)', '3', '(4, 8, 16)', '512',  'float32'],
    ['token IDs  [10, 20]',     '1', '(2,)',        '2',   'int64    ← integer indices'],
]

row_colors = ['#fef9e7', '#d6eaf8', '#fadbd8', '#d5f5e3', '#e8daef', '#fadbd8']
col_widths  = [3.2, 1.8, 2.4, 1.5, 3.7]
col_starts  = [0.1]
for w in col_widths[:-1]:
    col_starts.append(col_starts[-1] + w)

cell_h   = 0.52
header_y = 4.35
row_y0   = header_y - cell_h

# Header row
for h, x, w in zip(headers, col_starts, col_widths):
    ax.add_patch(plt.Rectangle((x, header_y), w - 0.06, cell_h,
                               fc='#1a5276', ec='white', lw=1.5))
    ax.text(x + (w-0.06)/2, header_y + cell_h/2, h,
            ha='center', va='center', fontsize=10, fontweight='bold', color='white')

# Data rows
for i, (row, rc) in enumerate(zip(rows, row_colors)):
    y = row_y0 - i * cell_h
    for j, (val, x, w) in enumerate(zip(row, col_starts, col_widths)):
        ax.add_patch(plt.Rectangle((x, y), w - 0.06, cell_h,
                                   fc=rc, ec='#d5d8dc', lw=0.8))
        # colour the dtype cell: red for int64, green for float32
        if j == 4:
            color = '#c0392b' if 'int64' in val else '#1e8449'
        else:
            color = '#1c2833'
        fw = 'bold' if j == 0 else 'normal'
        ax.text(x + (w-0.06)/2, y + cell_h/2, val,
                ha='center', va='center', fontsize=9, color=color, fontweight=fw)

# Rule callout below table
note_y = row_y0 - len(rows) * cell_h - 0.15
ax.text(0.1, note_y,
        'Rule:  decimal literals (1.0, 3.14)  →  float32     |     '
        'integer literals (1, 2, 3) or indices  →  int64',
        ha='left', va='top', fontsize=9.5, color='#1c2833',
        bbox=dict(boxstyle='round,pad=0.4', fc='#fef9e7', ec='#b7950b', lw=1.5))

ax.set_xlim(0, 13)
ax.set_ylim(note_y - 0.35, header_y + cell_h + 0.3)
plt.tight_layout()
plt.show()

## 4. Reading `(B, T, C)`

Suppose you see this in a Transformer notebook:

```python
B, T, C = 4, 8, 16
z = torch.randn(B, T, C)
```

Reading the shape `(4, 8, 16)` correctly is not optional — every chapter from here onward uses it. Here is what each axis means:

- `B = 4`: four sequences processed together in one batch
- `T = 8`: eight positions in each sequence (the context window)
- `C = 16`: sixteen numerical coordinates at each position

The mental model:

```text
Batch
│
├── sequence 0 → 8 positions, each a 16-dim vector
├── sequence 1 → 8 positions, each a 16-dim vector
├── sequence 2 → 8 positions, each a 16-dim vector
└── sequence 3 → 8 positions, each a 16-dim vector
```

The three natural slices are:

```python
z[0, 0, :]   # shape (16,)  — one token vector (C-dimensional)
z[0, :, :]   # shape (8,16) — one full sequence
z[0, :, 0]   # shape (8,)   — one coordinate traced across T positions
```

> **Which axis is the vector?**
> Any 1-D slice is mathematically a vector. In language-model notation, `z[b, t, :]` is the representation vector for position `t` because C is the feature axis — the coordinates that carry learned information. `z[b, :, c]` is also a vector, but it is better understood as one coordinate traced across T positions.
>
> The axes have distinct roles: **B** is the batch axis (sequences grouped for parallel GPU computation). **T** is the sequence axis (ordered positions in a context window — not a batch axis). **C** is the feature axis (coordinates learned during training). Attention later compares positions along T; learned transformations operate on and combine the feature coordinates in C.


In [ ]:
B, T, C = 4, 8, 16

# No embedding needed to understand the shape.
# A (B, T, C) tensor is just a 3-D block of numbers.
# We use torch.randn here — the shape is what matters, not where the values come from.
z = torch.randn(B, T, C)

print(f"z shape: {z.shape}")
print(f"  B={B} sequences in this batch")
print(f"  T={T} positions per sequence")
print(f"  C={C} values at each position  ← this is the vector dimension")

print(f"\nSlice z[0, 0, :] — one position's vector:")
print(f"  shape = {z[0, 0, :].shape}  ← a single {C}-dim vector")

print(f"\nSlice z[0, :, :] — one full sequence:")
print(f"  shape = {z[0, :, :].shape}  ← a T×C matrix (T vectors stacked)")

print(f"\nSlice z[0, :, 0] — one feature across all positions:")
print(f"  shape = {z[0, :, 0].shape}  ← a length-T signal for feature 0")

> **Aside — From `(B, T)` to `(B, T, C)`: Which Axis Is the Vector?**

This is a common source of confusion. The answer depends on which axis you treat as the feature dimension.

```text
                     C feature dimensions →
token / position 1   [ . . . . . . . . ]
token / position 2   [ . . . . . . . . ]
token / position 3   [ . . . . . . . . ]
...                  [ . . . . . . . . ]
token / position T   [ . . . . . . . . ]
          ↓
        T positions
```

- `z[b, t, :]` → one **token vector** of shape `(C,)` — the representation of position `t` in sequence `b`
- `z[b, :, :]` → one **sequence matrix** of shape `(T, C)` — all positions in sequence `b`
- `z[b, :, c]` → one **feature signal** of shape `(T,)` — how feature `c` varies across positions

> A vector is not inherently horizontal or vertical. Its role is determined by which axis is the feature dimension.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyArrowPatch

B_d, T_d, C_d = 4, 8, 16
slab_w  = 5.0
slab_h  = 3.5
dx, dy  = 0.35, 0.20
hi_t    = 2

fig, ax = plt.subplots(figsize=(11, 7))
ax.set_aspect('equal')
ax.axis('off')

# ── stacked slabs ────────────────────────────────────────────────────────────
for b in reversed(range(B_d)):
    ox, oy = b * dx, b * dy
    is_front = (b == 0)
    rect = mpatches.FancyBboxPatch(
        (ox, oy), slab_w, slab_h,
        boxstyle="square,pad=0",
        linewidth=1.8 if is_front else 0.9,
        edgecolor='#1a5276',
        facecolor='#d6eaf8' if is_front else '#aed6f1',
        alpha=0.85, zorder=B_d - b)
    ax.add_patch(rect)
    ax.text(ox + slab_w + 0.08, oy + slab_h / 2,
            f'b={b}', va='center', ha='left',
            fontsize=7.5, color='#1a5276', zorder=20)

# ── front-slab grid ──────────────────────────────────────────────────────────
for t in range(1, T_d):
    y = t * slab_h / T_d
    ax.plot([0, slab_w], [y, y], color='#5d8aa8', lw=0.4, alpha=0.55, zorder=10)
for c in range(1, C_d):
    x = c * slab_w / C_d
    ax.plot([x, x], [0, slab_h], color='#5d8aa8', lw=0.4, alpha=0.55, zorder=10)

# ── highlight z[0, hi_t, :] — red row (token vector) ────────────────────────
t_y0 = hi_t * slab_h / T_d
t_h  = slab_h / T_d
ax.add_patch(mpatches.Rectangle(
    (0, t_y0), slab_w, t_h,
    linewidth=2.2, edgecolor='#c0392b',
    facecolor='#f1948a', alpha=0.88, zorder=15))
ax.annotate(
    f'z[0, {hi_t}, :]  →  one C-dim token vector  (shape: ({C_d},))',
    xy=(slab_w, t_y0 + t_h / 2),
    xytext=(slab_w + 1.1, t_y0 + t_h / 2),
    fontsize=9, color='#c0392b', fontweight='bold', va='center',
    arrowprops=dict(arrowstyle='->', color='#c0392b', lw=1.4), zorder=25)

# ── highlight z[0, :, 3] — blue col (feature signal) ────────────────────────
c_col = 3
c_x0  = c_col * slab_w / C_d
c_w   = slab_w / C_d
ax.add_patch(mpatches.Rectangle(
    (c_x0, 0), c_w, slab_h,
    linewidth=2.2, edgecolor='#1e8bc3',
    facecolor='#85c1e9', alpha=0.70, zorder=14))
ax.annotate(
    f'z[0, :, {c_col}]  →  one feature across T positions  (shape: ({T_d},))',
    xy=(c_x0 + c_w / 2, slab_h),
    xytext=(c_x0 - 0.2, slab_h + 0.55),
    fontsize=9, color='#1a6fa8', fontweight='bold', ha='center',
    arrowprops=dict(arrowstyle='->', color='#1a6fa8', lw=1.4), zorder=25)

# ── outline z[0, :, :] — green border (sequence matrix) ─────────────────────
ax.add_patch(mpatches.Rectangle(
    (0, 0), slab_w, slab_h,
    linewidth=3.0, edgecolor='#1e8449',
    facecolor='none', zorder=22))
# label on LEFT side — avoids overlap with C-axis label at bottom
ax.annotate(
    f'z[0, :, :]  →  one T×C sequence matrix  (shape: ({T_d}, {C_d}))',
    xy=(0, slab_h * 0.18),
    xytext=(-2.5, slab_h * 0.18),
    fontsize=9, color='#1e8449', fontweight='bold',
    ha='right', va='center',
    arrowprops=dict(arrowstyle='->', color='#1e8449', lw=1.4), zorder=25)

# ── axis arrows and labels ───────────────────────────────────────────────────
def arrow(ax, x0, y0, x1, y1, color):
    ax.annotate('', xy=(x1, y1), xytext=(x0, y0),
                arrowprops=dict(arrowstyle='->', color=color, lw=1.8), zorder=30)

# C axis (horizontal, bottom)
arrow(ax, 0, -0.28, slab_w, -0.28, '#6c3483')
ax.text(slab_w / 2, -0.55,
        f'C = {C_d}  (embedding / feature dimension)',
        ha='center', fontsize=10, color='#6c3483', fontweight='bold')

# T axis (vertical, left)
arrow(ax, -0.28, 0, -0.28, slab_h, '#117a65')
ax.text(-0.55, slab_h / 2,
        f'T = {T_d}\n(token positions)',
        ha='center', va='center', fontsize=10, color='#117a65',
        fontweight='bold', rotation=90)

# B axis (depth, top-right)
arrow(ax, slab_w + (B_d-1)*dx + 0.15, slab_h + (B_d-1)*dy + 0.15,
          slab_w + 0.15,              slab_h + 0.15, '#7d6608')
ax.text(slab_w + (B_d-1)*dx/2 + 0.45,
        slab_h + (B_d-1)*dy/2 + 0.40,
        f'B = {B_d}  (batch)',
        ha='center', fontsize=10, color='#7d6608', fontweight='bold')

ax.set_title(
    f'Tensor shape  (B, T, C) = ({B_d}, {T_d}, {C_d})\n'
    'Each position in each sequence has a C-dimensional vector.',
    fontsize=12, fontweight='bold', pad=14, color='#1c2833')

ax.set_xlim(-5.5, slab_w + B_d*dx + 5.5)
ax.set_ylim(-0.85, slab_h + B_d*dy + 1.2)
plt.tight_layout()
plt.show()

In [ ]:
B, T, C = 4, 8, 16
z = torch.randn(B, T, C)   # random values — shape is what we are studying

# Three slices — three different views of the same tensor
token_vector    = z[0, 0, :]   # one position, all C features
sequence_matrix = z[0, :, :]   # one sequence, all T positions × C features
feature_signal  = z[0, :, 0]   # one feature dimension, traced over T positions

print(f"z shape:              {z.shape}")
print(f"z[0, 0, :]  shape = {token_vector.shape}    ← one {C}-dim vector (a single position)")
print(f"z[0, :, :]  shape = {sequence_matrix.shape}  ← T×C matrix (all positions in one sequence)")
print(f"z[0, :, 0]  shape = {feature_signal.shape}    ← feature 0 traced across {T} positions")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle('Three slices from z  shape=(B, T, C)', fontsize=12, fontweight='bold')

B_d, T_d, C_d = 4, 8, 16
configs = [
    ('z[b, t, :]', f'shape: ({C_d},)', 'One token\'s vector\n(one row = one position)',
     1, C_d, '#f1948a', '#c0392b'),
    ('z[b, :, :]', f'shape: ({T_d}, {C_d})', 'Sequence matrix\n(all positions, all features)',
     T_d, C_d, '#85c1e9', '#1a6fa8'),
    ('z[b, :, c]', f'shape: ({T_d},)', 'Feature signal\n(one feature across positions)',
     T_d, 1, '#a9dfbf', '#1e8449'),
]

for ax, (expr, shp, desc, rows, cols, fc, ec) in zip(axes, configs):
    ax.axis('off'); ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.set_title(f'`{expr}`\n{shp}', fontsize=10, fontweight='bold', color=ec)

    # Draw mini grid
    cw = min(0.80 / cols, 0.15)
    ch = min(0.40 / rows, 0.12)
    sx = 0.5 - cols * (cw + 0.01) / 2
    sy = 0.35

    for r in range(rows):
        for c in range(cols):
            ax.add_patch(plt.Rectangle(
                (sx + c * (cw + 0.01), sy + r * (ch + 0.01)),
                cw, ch, fc=fc, ec=ec, lw=0.8, alpha=0.85
            ))

    ax.text(0.5, 0.28, shp, ha='center', fontsize=9, color=ec, fontweight='bold')
    ax.text(0.5, 0.13, desc, ha='center', fontsize=8.5, color='#1c2833',
            va='center', style='italic')

    ax.add_patch(plt.Rectangle((0.02, 0.05), 0.96, 0.90,
                               fc='none', ec=ec, lw=2.0, ls='--'))

plt.tight_layout()
plt.show()

## 5. Coordinates, Magnitude, and Direction

For a 2-D vector `v = [3, 4]`, the coordinates tell you where it points in space.

**Magnitude** (norm) = length of the vector:

$$\|v\| = \sqrt{3^2 + 4^2} = 5$$

The magnitude captures *how large* a vector is. The direction captures *which way* it points.

These two ideas are used everywhere:
- **Norms** → measure the size of a representation
- **Direction** → cosine similarity compares directions, ignoring magnitude

> **Aside — L2 norm vs LayerNorm.** Do not confuse this L2 norm with LayerNorm. LayerNorm normalises a vector using its feature-wise mean and variance, not its L2 magnitude. A vector can have a small L2 norm but a large variance across its components, or vice versa. Chapter 10 introduces LayerNorm in its architectural context.

In [ ]:
v = torch.tensor([3.0, 4.0])
magnitude = torch.linalg.vector_norm(v)
print(f"v          = {v.tolist()}")
print(f"||v||      = {magnitude:.4f}  (expected 5.0)")

# Normalise to unit length (direction only)
v_unit = v / magnitude
print(f"v_unit     = {v_unit.tolist()}")
print(f"||v_unit|| = {torch.linalg.vector_norm(v_unit):.4f}  (always 1.0)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Magnitude and Direction', fontsize=13, fontweight='bold')

# ── Panel 1: magnitude as length ─────────────────────────────────────────────
ax = axes[0]
ax.set_xlim(-0.5, 4.5); ax.set_ylim(-0.5, 5.2)
ax.set_aspect('equal'); ax.grid(True, alpha=0.3)
ax.axhline(0, color='gray', lw=0.8); ax.axvline(0, color='gray', lw=0.8)
ax.set_title('Magnitude = length of the vector', fontsize=11)

v = [3, 4]
ax.annotate('', xy=v, xytext=[0,0],
            arrowprops=dict(arrowstyle='->', color='#1a5276', lw=2.5))
ax.text(v[0]/2 - 0.45, v[1]/2 + 0.15, '||v|| = 5', fontsize=12,
        color='#1a5276', fontweight='bold')

# dashed right-angle triangle
ax.plot([0, 3, 3], [0, 0, 4], color='#922b21', lw=1.2, ls='--')
ax.text(1.5, -0.35, '3', ha='center', fontsize=11, color='#922b21')
ax.text(3.25, 2.0, '4', ha='left', fontsize=11, color='#922b21')
ax.text(v[0]+0.15, v[1]+0.15, 'v = [3, 4]', fontsize=11, color='#1a5276')
ax.set_xlabel('x'); ax.set_ylabel('y')

# ── Panel 2: unit vector — same direction, length 1 ──────────────────────────
ax = axes[1]
ax.set_xlim(-0.5, 4.5); ax.set_ylim(-0.5, 5.2)
ax.set_aspect('equal'); ax.grid(True, alpha=0.3)
ax.axhline(0, color='gray', lw=0.8); ax.axvline(0, color='gray', lw=0.8)
ax.set_title('Direction = unit vector (magnitude = 1)', fontsize=11)

import numpy as np
v_arr = np.array([3.0, 4.0])
v_unit = v_arr / np.linalg.norm(v_arr)

ax.annotate('', xy=v_arr, xytext=[0,0],
            arrowprops=dict(arrowstyle='->', color='#aab7b8', lw=1.5, linestyle='dashed'))
ax.text(v_arr[0]+0.1, v_arr[1]+0.1, 'v (length 5)', fontsize=9, color='#aab7b8')

ax.annotate('', xy=v_unit, xytext=[0,0],
            arrowprops=dict(arrowstyle='->', color='#c0392b', lw=2.5))
ax.text(v_unit[0]+0.1, v_unit[1]+0.1, 'v̂ = v / ||v||\n(length 1)',
        fontsize=10, color='#c0392b', fontweight='bold')

theta = np.linspace(0, np.pi/2, 100)
ax.plot(0.5*np.cos(theta), 0.5*np.sin(theta), color='#1a5276', lw=1.0, ls='--')
angle_deg = np.degrees(np.arctan2(v_arr[1], v_arr[0]))
ax.text(0.6*np.cos(np.radians(angle_deg/2)), 0.6*np.sin(np.radians(angle_deg/2)),
        f'{angle_deg:.0f}°', fontsize=9, color='#1a5276')
ax.set_xlabel('x'); ax.set_ylabel('y')

plt.tight_layout()
plt.show()

## 6. Vector Addition and Scalar Multiplication

**Addition:** `u + v` — add element-wise. Moves in two directions at once.

**Scalar multiplication:** `α * v` — stretch or shrink the vector.

Later connections in the book:
- **Residual connections:** `x = x + F(x)` — vector addition
- **Attention output:** weighted combination of value vectors — scalar multiplication + addition
- **Gradient updates:** `w = w - lr * grad` — scalar multiplication + subtraction

In [ ]:
u = torch.tensor([1.0, 2.0, 3.0])
v = torch.tensor([4.0, 5.0, 6.0])

print(f"u         = {u.tolist()}")
print(f"v         = {v.tolist()}")
print(f"u + v     = {(u + v).tolist()}")
print(f"2.0 * v   = {(2.0 * v).tolist()}")
print(f"u + 0.5*v = {(u + 0.5 * v).tolist()}")

# Residual connection preview: input + transformed_input
x      = torch.tensor([0.5, -0.3, 0.8])
F_x    = torch.tensor([0.1,  0.2, -0.1])  # some transformation output
output = x + F_x
print(f"\nResidual: x + F(x) = {output.tolist()}")

In [ ]:
import numpy as np
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
fig.suptitle('Vector Addition and Scalar Multiplication', fontsize=13, fontweight='bold')

u = np.array([1.0, 2.0]); v = np.array([3.0, 1.0])
origin = np.array([0.0, 0.0])

# ── Panel 1: parallelogram rule for u + v ────────────────────────────────────
ax = axes[0]
ax.set_xlim(-0.3, 5.2); ax.set_ylim(-0.3, 3.8)
ax.set_aspect('equal'); ax.grid(True, alpha=0.3)
ax.axhline(0, color='gray', lw=0.8); ax.axvline(0, color='gray', lw=0.8)
ax.set_title('u + v  (parallelogram rule)', fontsize=11)

ax.annotate('', xy=u, xytext=origin,
            arrowprops=dict(arrowstyle='->', color='#1a5276', lw=2.2))
ax.text(u[0]/2 - 0.3, u[1]/2 + 0.1, 'u', fontsize=13, color='#1a5276', fontweight='bold')

ax.annotate('', xy=v, xytext=origin,
            arrowprops=dict(arrowstyle='->', color='#922b21', lw=2.2))
ax.text(v[0]/2 + 0.1, v[1]/2 - 0.25, 'v', fontsize=13, color='#922b21', fontweight='bold')

ax.annotate('', xy=u+v, xytext=v,
            arrowprops=dict(arrowstyle='->', color='#1a5276', lw=1.2, linestyle='dashed'))
ax.annotate('', xy=u+v, xytext=u,
            arrowprops=dict(arrowstyle='->', color='#922b21', lw=1.2, linestyle='dashed'))

ax.annotate('', xy=u+v, xytext=origin,
            arrowprops=dict(arrowstyle='->', color='#117a65', lw=2.8))
ax.text((u+v)[0]/2 - 0.15, (u+v)[1]/2 + 0.18,
        'u + v', fontsize=12, color='#117a65', fontweight='bold')
ax.set_xlabel('x'); ax.set_ylabel('y')

# ── Panel 2: scalar multiplication ───────────────────────────────────────────
ax = axes[1]
ax.set_xlim(-4.5, 4.5); ax.set_ylim(-2.5, 3.2)
ax.set_aspect('equal'); ax.grid(True, alpha=0.3)
ax.axhline(0, color='gray', lw=0.8); ax.axvline(0, color='gray', lw=0.8)
ax.set_title('α × v  (scalar multiplication)', fontsize=11)

base = np.array([1.5, 0.8])

# perpendicular direction to base — for staggering labels
perp = np.array([-base[1], base[0]])
perp = perp / np.linalg.norm(perp)

# (alpha, color, label, perp_offset_for_label, along_offset)
configs = [
    ( 2.0, '#8e44ad', '2.0 × v',           0.55,  0.12),
    ( 1.0, '#1a5276', '1.0 × v  (original)', 0.20, 0.12),
    ( 0.5, '#117a65', '0.5 × v',           -0.25,  0.12),
    (-1.0, '#922b21', '−1.0 × v  (flipped)', 0.35, 0.12),
]

for alpha, color, label, perp_off, along_off in configs:
    vec = alpha * base
    ax.annotate('', xy=vec, xytext=origin,
                arrowprops=dict(arrowstyle='->', color=color, lw=2.0))

    # place label offset perpendicularly to avoid overlap
    tip_dir = vec / (np.linalg.norm(vec) + 1e-9)
    lx = vec[0] + tip_dir[0] * along_off + perp[0] * perp_off
    ly = vec[1] + tip_dir[1] * along_off + perp[1] * perp_off
    ax.text(lx, ly, label, fontsize=9, color=color, fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.18', fc='white', ec=color, lw=0.8, alpha=0.85))

ax.set_xlabel('x'); ax.set_ylabel('y')

plt.tight_layout()
plt.show()

## 7. The Dot Product

The dot product multiplies corresponding coordinates and sums the results:

$$u \cdot v = u_1 v_1 + u_2 v_2 + \cdots + u_n v_n$$

**Geometric intuition:**
- Large positive → vectors point in the same direction (aligned)
- Near zero → vectors are roughly perpendicular (orthogonal in the geometric sense)
- Negative → vectors point in opposite directions

Magnitude affects the raw dot product. Cosine similarity (Section 7) normalises this away.

---

> **Research Connection — Vaswani et al. (2017), *Attention Is All You Need***
>
> The Transformer later compares query and key vectors using a dot product. We have learned the primitive here; the self-attention chapter will build the mechanism. At this point all we need is the operation itself.

In [ ]:
u = torch.tensor([1.0, 2.0, 3.0])
v = torch.tensor([4.0, 5.0, 6.0])
w = torch.tensor([-1.0, 0.0, 1.0])

print(f"u = {u.tolist()}")
print(f"v = {v.tolist()}")
print(f"w = {w.tolist()}")

# Compute
dot_uv = torch.dot(u, v)
dot_uw = torch.dot(u, w)

print(f"\nu · v = {dot_uv.item():.2f}  ← large positive: same direction")
print(f"u · w = {dot_uw.item():.2f}   ← near zero: weakly aligned")

# Manual verification
manual = sum(a * b for a, b in zip(u.tolist(), v.tolist()))
print(f"\nManual u·v = {manual}  (matches torch.dot)")
print(f"\nForeshadowing: attention computes Q · K for every (query, key) pair.")
print(f"High dot product → token attends to that position.")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

u = np.array([1.0, 2.0, 3.0])
v = np.array([4.0, 5.0, 6.0])
w = np.array([-1.0, 0.0, 1.0])

fig, (ax_left, ax_right) = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle("Dot Product — element-wise computation and geometric alignment",
             fontsize=13, fontweight="bold")

# ─── LEFT: step-by-step computation
ax_left.axis("off")
ax_left.set_title("Computation  (u · v  and  u · w)", fontsize=11, fontweight="bold", pad=10)

def draw_computation(ax, a_vec, b_vec, name_a, name_b, y_top, header_clr):
    products = [float(ai * bi) for ai, bi in zip(a_vec, b_vec)]
    total = sum(products)

    ax.text(0.03, y_top, f"{name_a} · {name_b}",
            fontsize=14, fontweight="bold", color=header_clr,
            transform=ax.transAxes, va="top")

    term_colors = ["#1a5276", "#117a65", "#6c3483"]
    xs = [0.03, 0.38, 0.69]
    for i, (ai, bi, p, xi) in enumerate(zip(a_vec, b_vec, products, xs)):
        tc = term_colors[i]
        sign = "+" if p >= 0 else ""
        label = f"{ai:+.1f} × {bi:+.1f}\n   = {sign}{p:.1f}"
        ax.text(xi, y_top - 0.09, label,
                fontsize=10.5, color=tc, fontweight="bold",
                ha="left", va="top", transform=ax.transAxes,
                bbox=dict(boxstyle="round,pad=0.35", fc=tc, ec=tc, alpha=0.12))
        if i < 2:
            mid_x = (xs[i] + xs[i + 1]) / 2 + 0.04
            ax.text(mid_x, y_top - 0.13, "+", fontsize=15, color="#7f8c8d",
                    ha="center", va="top", transform=ax.transAxes)

    s_clr = "#1e8449" if total > 5 else ("#ca6f1e" if total >= 0 else "#922b21")
    parts = " + ".join(f"{p:.1f}" for p in products)
    ax.text(0.03, y_top - 0.30,
            f"= {parts}  =  {total:.1f}",
            fontsize=11.5, fontweight="bold", color=s_clr,
            transform=ax.transAxes, va="top",
            bbox=dict(boxstyle="round,pad=0.4", fc=s_clr, ec=s_clr, alpha=0.13))

    if total > 5:
        note = "Large positive → vectors are closely aligned"
    elif total >= 0:
        note = "Small positive → vectors are weakly aligned"
    else:
        note = "Negative → vectors point in opposing directions"
    ax.text(0.03, y_top - 0.41, note,
            fontsize=9.5, color=s_clr, style="italic",
            transform=ax.transAxes, va="top")

draw_computation(ax_left, u, v, "u", "v", 0.94, "#1a5276")
ax_left.plot([0.02, 0.98], [0.50, 0.50], color="#bdc3c7", lw=1.2,
             transform=ax_left.transAxes)
draw_computation(ax_left, u, w, "u", "w", 0.46, "#c0392b")

# ─── RIGHT: direction / alignment panel
ax_right.set_title("Geometric direction (2D projection of unit vectors)",
                   fontsize=11, fontweight="bold", pad=10)
ax_right.set_xlim(-1.5, 1.5)
ax_right.set_ylim(-0.5, 1.5)
ax_right.set_aspect("equal")
ax_right.axhline(0, color="#bdc3c7", lw=0.8)
ax_right.axvline(0, color="#bdc3c7", lw=0.8)
ax_right.grid(True, alpha=0.2)

origin = np.array([0.0, 0.0])

def proj_unit2d(vec):
    uhat = vec / np.linalg.norm(vec)
    xy = uhat[:2]
    n = np.linalg.norm(xy)
    return xy / n if n > 1e-9 else np.array([1.0, 0.0])

u2d = proj_unit2d(u)   # ≈ [0.447,  0.894] — upper-right
v2d = proj_unit2d(v)   # ≈ [0.625,  0.781] — upper-right, close to u
w2d = proj_unit2d(w)   # =  [-1.0,   0.0 ] — straight left

for pt, lbl, clr in [(u2d, "u", "#1a5276"), (v2d, "v", "#117a65"), (w2d, "w", "#c0392b")]:
    ep = pt * 0.92
    ax_right.annotate("", xy=ep, xytext=origin,
                      arrowprops=dict(arrowstyle="->", color=clr, lw=2.5))
    off = pt * 0.15
    ax_right.text(ep[0] + off[0], ep[1] + off[1], lbl,
                  fontsize=13, color=clr, fontweight="bold", ha="center",
                  bbox=dict(boxstyle="round,pad=0.3", fc="white", ec=clr, lw=1.2, alpha=0.92))

def angle3d(a, b):
    return np.degrees(np.arccos(np.clip(
        np.dot(a / np.linalg.norm(a), b / np.linalg.norm(b)), -1.0, 1.0)))

theta_uv = angle3d(u, v)   # ≈12.7° — nearly aligned
theta_uw = angle3d(u, w)   # ≈67.8° — clearly different direction

ang_u = np.degrees(np.arctan2(u2d[1], u2d[0]))   # ≈  63.4°
ang_v = np.degrees(np.arctan2(v2d[1], v2d[0]))   # ≈  51.3°
ang_w = np.degrees(np.arctan2(w2d[1], w2d[0]))   # = 180.0°

r_uv = 0.48
arc_uv = mpatches.Arc(origin, 2 * r_uv, 2 * r_uv, angle=0,
                       theta1=min(ang_u, ang_v), theta2=max(ang_u, ang_v),
                       color="#117a65", lw=2.2, linestyle="--")
ax_right.add_patch(arc_uv)
mid_uv = np.radians((ang_u + ang_v) / 2)
ax_right.text(r_uv * 1.40 * np.cos(mid_uv), r_uv * 1.40 * np.sin(mid_uv),
              f"θ ≈ {theta_uv:.0f}°\nu·v = {int(np.dot(u, v))}",
              fontsize=9, color="#117a65", fontweight="bold", ha="center",
              bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="#117a65", lw=1))

r_uw = 0.70
arc_uw = mpatches.Arc(origin, 2 * r_uw, 2 * r_uw, angle=0,
                       theta1=min(ang_u, ang_w), theta2=max(ang_u, ang_w),
                       color="#c0392b", lw=2.2, linestyle="--")
ax_right.add_patch(arc_uw)
mid_uw = np.radians((ang_u + ang_w) / 2)
ax_right.text(r_uw * 1.28 * np.cos(mid_uw), r_uw * 1.28 * np.sin(mid_uw),
              f"θ ≈ {theta_uw:.0f}°\nu·w = {int(np.dot(u, w))}",
              fontsize=9, color="#c0392b", fontweight="bold", ha="center",
              bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="#c0392b", lw=1))

ax_right.set_xlabel("x")
ax_right.set_ylabel("y")
ax_right.text(0.5, -0.07,
              "Arrows: 2D projection of unit vectors.  Angle labels: true 3D values.",
              ha="center", fontsize=8, color="#95a5a6", transform=ax_right.transAxes)

plt.tight_layout()
plt.show()

## 8. Distance and Similarity

### Euclidean Distance
How far apart are two points in space?

$$\|u - v\| = \sqrt{\sum_i (u_i - v_i)^2}$$

### Cosine Similarity
How similar are two **directions**, regardless of magnitude?

$$\cos(u, v) = \frac{u \cdot v}{\|u\| \|v\|}$$

Cosine similarity is bounded between −1 and +1.

**Important:** high cosine similarity does not mean "same meaning." It means the two vectors point in similar directions *in the space they were built in*. Whether that direction is semantically, syntactically, or predictively useful depends entirely on how the vectors were produced.

**Forward connections:** embeddings, semantic search, retrieval, RAG.

---

> **Research Connection — Mikolov et al. (2013), *Efficient Estimation of Word Representations in Vector Space***
>
> Trained vector spaces can develop systematic geometric regularities measurable through cosine similarity and distance. Chapter 2 gives us the measurement tools; Chapter 3 will ask how learned representations acquire such structure — and we will run the experiments ourselves.

In [ ]:
u = torch.tensor([1.0, 2.0, 3.0])
v = torch.tensor([2.0, 4.0, 6.0])  # same direction, 2x magnitude
w = torch.tensor([1.0, 0.0, -1.0]) # different direction

# --- Euclidean distance ---
# vector_norm(x) computes the L2 norm: sqrt(x1^2 + x2^2 + ... + xn^2)
# vector_norm(u - v) = length of the difference vector = Euclidean distance
dist_uv = torch.linalg.vector_norm(u - v)
dist_uw = torch.linalg.vector_norm(u - w)
print(f"u - v = {u - v}")
print(f"vector_norm(u - v) = sqrt({(u-v).pow(2).sum():.1f}) = {dist_uv:.4f}")
print(f"Euclidean distance u<->v: {dist_uv:.4f}  (far apart because v is 2x bigger)")
print(f"Euclidean distance u<->w: {dist_uw:.4f}")

# --- Cosine similarity ---
# F.cosine_similarity is designed for BATCHED inputs: shape (batch, features).
# Our vectors are 1D with shape (3,) — a bare vector, not a batch.
# unsqueeze(0) inserts a new axis at position 0:
#   u.shape          = (3,)   — 1D vector
#   u.unsqueeze(0).shape = (1, 3) — "batch" of 1 vector
# Without unsqueeze, cosine_similarity would operate on the wrong axis.
# Alternative that needs no reshaping: torch.dot(u,v) / (norm(u) * norm(v))
cos_uv = F.cosine_similarity(u.unsqueeze(0), v.unsqueeze(0)).item()
cos_uw = F.cosine_similarity(u.unsqueeze(0), w.unsqueeze(0)).item()
print(f"Cosine similarity u<->v: {cos_uv:.4f}  (identical direction -> 1.0)")
print(f"Cosine similarity u<->w: {cos_uw:.4f}  (different direction)")

print("Key insight: u and v are FAR in Euclidean space (v = 2*u, so ||u-v|| > 0)")
print("but IDENTICAL in cosine (same direction) — cosine ignores magnitude entirely.")

In [ ]:
import numpy as np
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Euclidean Distance vs Cosine Similarity', fontsize=13, fontweight='bold')

origin = np.array([0.0, 0.0])

# ── Panel 1: same direction, different magnitude ──────────────────────────────
ax = axes[0]
ax.set_xlim(-0.5, 7); ax.set_ylim(-0.5, 7)
ax.set_aspect('equal'); ax.grid(True, alpha=0.3)
ax.axhline(0, color='gray', lw=0.8); ax.axvline(0, color='gray', lw=0.8)
ax.set_title('Same direction, different magnitude', fontsize=11)

u1 = np.array([1.0, 2.0]); v1 = np.array([2.0, 4.0])
ax.annotate('', xy=u1, xytext=origin, arrowprops=dict(arrowstyle='->', color='#1a5276', lw=2.5))
ax.text(u1[0]+0.1, u1[1]+0.1, 'u', fontsize=12, color='#1a5276', fontweight='bold')
ax.annotate('', xy=v1, xytext=origin, arrowprops=dict(arrowstyle='->', color='#c0392b', lw=2.5))
ax.text(v1[0]+0.1, v1[1]+0.1, 'v = 2u', fontsize=12, color='#c0392b', fontweight='bold')

dist = np.linalg.norm(u1 - v1)
cos  = np.dot(u1, v1) / (np.linalg.norm(u1) * np.linalg.norm(v1))
ax.plot([u1[0], v1[0]], [u1[1], v1[1]], 'k--', lw=1.5)
ax.text(3.5, 1.0, f'Euclidean dist = {dist:.2f}\nCosine sim      = {cos:.2f}',
        fontsize=10, color='#1c2833',
        bbox=dict(boxstyle='round,pad=0.4', fc='#fef9e7', ec='#b7950b', lw=1.5))
ax.set_xlabel('x'); ax.set_ylabel('y')

# ── Panel 2: different directions ─────────────────────────────────────────────
ax = axes[1]
ax.set_xlim(-0.5, 4); ax.set_ylim(-3, 3)
ax.set_aspect('equal'); ax.grid(True, alpha=0.3)
ax.axhline(0, color='gray', lw=0.8); ax.axvline(0, color='gray', lw=0.8)
ax.set_title('Different directions', fontsize=11)

u2 = np.array([1.0, 2.0]); w2 = np.array([1.0, -2.0])
ax.annotate('', xy=u2, xytext=origin, arrowprops=dict(arrowstyle='->', color='#1a5276', lw=2.5))
ax.text(u2[0]+0.1, u2[1]+0.1, 'u', fontsize=12, color='#1a5276', fontweight='bold')
ax.annotate('', xy=w2, xytext=origin, arrowprops=dict(arrowstyle='->', color='#8e44ad', lw=2.5))
ax.text(w2[0]+0.1, w2[1]-0.3, 'w', fontsize=12, color='#8e44ad', fontweight='bold')

dist2 = np.linalg.norm(u2 - w2)
cos2  = np.dot(u2, w2) / (np.linalg.norm(u2) * np.linalg.norm(w2))
ax.plot([u2[0], w2[0]], [u2[1], w2[1]], 'k--', lw=1.5)
ax.text(1.6, 0.2, f'Euclidean dist = {dist2:.2f}\nCosine sim      = {cos2:.2f}',
        fontsize=10, color='#1c2833',
        bbox=dict(boxstyle='round,pad=0.4', fc='#fef9e7', ec='#b7950b', lw=1.5))
ax.set_xlabel('x'); ax.set_ylabel('y')

plt.tight_layout()
plt.show()

## 9. Matrices as Collections of Vectors

A matrix can be read in two ways:
- **Row view:** each row is a vector
- **Column view:** each column is a vector
- **Transformation view:** multiplying a vector by a matrix produces a new vector

A matrix with V rows and C columns is a collection of V vectors, each of length C. In mathematical notation:

$$W \in \mathbb{R}^{V \times C}$$

Reading this left to right:

| Symbol | Meaning |
|--------|---------|
| $W$ | the name of the matrix (a weight table with V rows and C columns) |
| $\in$ | "is an element of" — W belongs to this set |
| $\mathbb{R}$ | the set of all real numbers (floating-point values) |
| $\mathbb{R}^{V \times C}$ | the set of all matrices with $V$ rows and $C$ columns, each entry a real number |
| $V$ | vocabulary size — one row per token (95 in Chapter 1) |
| $C$ | feature dimension — how many numbers describe each token |

In plain English: **W is a matrix of real numbers with V rows and C columns.**

In PyTorch: `W.shape == (V, C)` — `V` rows, each a `C`-dimensional float vector.

Every row is a `C`-dimensional vector. Row selection by integer index gives you that vector.

The lookup table from Chapter 1 had the same row-selection pattern, but its shape was 95 × 95: one row per character and 95 logits in every row. Here `C = 8` is only a mathematical example that lets us study a `V × C` matrix. Chapter 3 will return to the distinction between `V` and `C`. Other parameter matrices such as Q, K, and V are used differently: they transform via full matrix multiplication rather than selecting a row.

In [ ]:
V = vocab_size   # 95 characters
C = 8            # 8 numbers per row

# A plain weight matrix: V rows, C columns
W = torch.randn(V, C)

print(f'Weight matrix shape: {W.shape}')
print(f'  {W.shape[0]} rows -- one per character in the vocabulary')
print(f'  {W.shape[1]} columns -- C numbers in each row')

# Each row is a C-dimensional vector
for i, ch in enumerate(chars[:4]):
    row = W[i]
    print(f"  char '{ch}' (id={i}): row shape={row.shape}")

# Row selection by index
idx = stoi['h']
row = W[idx]
print(f"\nRow for h (id={idx}): shape={row.shape}")
print(f"  first 4 values: {row.detach()[:4].tolist()}")

print(
    f"\nThe Chapter 1 lookup table used the same row-selection pattern, "
    f"but had shape ({V}, {V})."
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Matrices as Collections of Vectors', fontsize=13, fontweight='bold', y=1.02)

V_d, C_d = 5, 4
row_colors = ['#d6eaf8', '#d5f5e3', '#fdebd0', '#e8daef', '#fadbd8']
row_labels = ["token 0  'a'", "token 1  'b'", "token 2  ' '", "token 3  '!'", f"token {V_d-1} ..."]
cw, ch, gap = 0.9, 0.42, 0.05
sx, sy = 1.1, 0.35

for panel, (title, highlight, arrow_kw, note_kw) in enumerate([
    ('Row view — each row is one token\'s vector',
     dict(rect=(sx-0.05, sy + 2*(ch+gap) - 0.04, V_d*(cw+gap)-gap+0.1, ch+0.08),
          ec='#c0392b', label_xy=(sx + V_d*(cw+gap)-gap+0.12, sy + 2*(ch+gap) + ch/2 - 0.04),
          label_text="← row 2\n= vector\nfor ' '", color='#c0392b'),
     None, None),
    ('Column view — each column is one feature\nacross all tokens',
     dict(rect=(sx + 1*(cw+gap) - 0.04, sy - 0.04, cw+0.08, V_d*(ch+gap)-gap+0.08),
          ec='#2980b9', label_xy=(sx + 1*(cw+gap) + cw/2, sy + V_d*(ch+gap)-gap+0.12),
          label_text="↑ col 1\n= feature\nacross\nall tokens", color='#2980b9'),
     None, None),
]):
    ax = axes[panel]
    ax.set_xlim(0, 6); ax.set_ylim(0, 3.2)
    ax.axis('off')
    ax.set_title(title, fontsize=10)

    # Draw grid
    for i, (rc, rl) in enumerate(zip(row_colors, row_labels)):
        y = sy + (V_d - 1 - i) * (ch + gap)
        for j in range(C_d):
            x = sx + j * (cw + gap)
            ax.add_patch(plt.Rectangle((x, y), cw, ch, fc=rc, ec='#7f8c8d', lw=1.0))
            ax.text(x + cw/2, y + ch/2, f'e{i}{j}',
                    ha='center', va='center', fontsize=8.5, color='#2c3e50')
        ax.text(sx - 0.08, y + ch/2, rl, ha='right', va='center', fontsize=8, color='#34495e')

    # Column header labels
    for j in range(C_d):
        ax.text(sx + j*(cw+gap) + cw/2, sy + V_d*(ch+gap)-gap + 0.06,
                f'c={j}', ha='center', fontsize=8.5, color='#555', fontweight='bold')

    # Highlight
    h = highlight
    ax.add_patch(plt.Rectangle(h['rect'][:2], h['rect'][2], h['rect'][3],
                               fc='none', ec=h['ec'], lw=2.5, ls='--'))
    ax.text(h['label_xy'][0] + 0.05, h['label_xy'][1],
            h['label_text'], ha='left', va='center',
            fontsize=8.5, color=h['color'], fontweight='bold')

    # Matrix label
    ax.text(sx + C_d*(cw+gap)/2 - gap/2, 0.12,
            f'E  ∈  ℝ^({V_d} × {C_d})     V rows × C cols',
            ha='center', fontsize=10, color='#1c2833',
            bbox=dict(boxstyle='round,pad=0.3', fc='#f8f9fa', ec='#aab7b8', lw=1.0))

plt.tight_layout()
plt.show()

## 10. Matrix Multiplication as Transformation

$$y = x W$$

A vector `x` of dimension `in` enters. A weight matrix `W` of shape `(in, out)` transforms it. A new vector `y` of dimension `out` comes out.

Shape rule: `(in,) @ (in, out) → (out,)`

**Forward connections:**
- Output projection: `(C,) @ (C, vocab_size) → (vocab_size,)` — produces logits
- Q/K/V projections in attention
- Feed-forward layers
- LoRA: `x @ A @ B` — two matrix multiplications in sequence

In [ ]:
in_dim  = 8
out_dim = 4

x = torch.randn(in_dim)          # one token vector
W = torch.randn(in_dim, out_dim) # weight matrix
y = x @ W                        # matrix multiply

print(f"Input  x: shape={x.shape}")
print(f"Weight W: shape={W.shape}")
print(f"Output y: shape={y.shape}")
print(f"\nShape trace: ({in_dim},) @ ({in_dim}, {out_dim}) → ({out_dim},)")

# Batched: apply same transformation to a full sequence
T = 6
X = torch.randn(T, in_dim)  # sequence of T token vectors
Y = X @ W                   # transforms every token independently
print(f"\nBatched: ({T}, {in_dim}) @ ({in_dim}, {out_dim}) → {Y.shape}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

cw = ch = 0.50
c_x  = '#1a5276'
c_W  = '#922b21'
c_y  = '#117a65'
c_hl = '#e67e22'

fig, (ax_top, ax_bot) = plt.subplots(
    2, 1, figsize=(14, 8), gridspec_kw={'height_ratios': [3, 1]})
fig.suptitle('Matrix Multiplication  —  (8,) @ (8, 4)  →  (4,)',
             fontsize=13, fontweight='bold')

ax_top.axis('off')
ax_top.set_xlim(0, 14)
ax_top.set_ylim(0, 7)

# W spans y=1.5 to 5.5 (8 rows × 0.5 = 4.0 tall)
W_y0_bot   = 1.5
W_y0_top   = W_y0_bot + 8*ch    # 5.5
W_center_y = W_y0_bot + 4*ch    # 3.5  — vertical midpoint

# x, @, =, y are all centred at W_center_y
row_y0 = W_center_y - ch/2      # 3.25  — bottom of the 1-row strips

# ── x: 1×8 HORIZONTAL row vector, left of @, centred on W ──────────────────
x_x0 = 0.30
for j in range(8):
    rect = plt.Rectangle((x_x0 + j*cw, row_y0), cw, ch,
                          fc=c_x, ec='white', lw=1.5, alpha=0.88)
    ax_top.add_patch(rect)
    ax_top.text(x_x0 + j*cw + cw/2, row_y0 + ch/2,
                f'x[{j}]', ha='center', va='center',
                fontsize=7.5, color='white', fontweight='bold')

# x labels
x_mid = x_x0 + 4*cw
ax_top.text(x_mid, row_y0 + ch + 0.20,
            'x   —   shape (8,)   —   row vector',
            ha='center', va='bottom', fontsize=11, color=c_x, fontweight='bold')
ax_top.text(x_mid, row_y0 - 0.18,
            '8 values  (must match 8 rows of W)',
            ha='center', va='top', fontsize=8.5, color='#7f8c8d', style='italic')

# ── @ operator ──────────────────────────────────────────────────────────────
at_x = x_x0 + 8*cw + 0.25
ax_top.text(at_x + 0.15, W_center_y, '@',
            ha='center', va='center', fontsize=24, color='#555', fontweight='bold')

# ── W: 8 rows × 4 cols, row 0 at TOP ────────────────────────────────────────
W_x0 = at_x + 0.55
for i in range(8):                    # i=0 → top row
    for j in range(4):
        fc = c_hl if j == 0 else c_W
        cell_y = W_y0_bot + (7 - i)*ch   # row 0 at top
        rect = plt.Rectangle((W_x0 + j*cw, cell_y), cw, ch,
                              fc=fc, ec='white', lw=1.5, alpha=0.88)
        ax_top.add_patch(rect)
        ax_top.text(W_x0 + j*cw + cw/2, cell_y + ch/2,
                    f'w[{i},{j}]', ha='center', va='center',
                    fontsize=5.8, color='white', fontweight='bold')

# column headers (above W)
for j in range(4):
    clr = c_hl if j == 0 else '#c0392b'
    ax_top.text(W_x0 + j*cw + cw/2, W_y0_top + 0.06,
                f'col {j}', ha='center', va='bottom', fontsize=8, color=clr, fontweight='bold')

# row index labels (left of W, 0 at top)
for i in range(8):
    cell_y = W_y0_bot + (7 - i)*ch
    ax_top.text(W_x0 - 0.08, cell_y + ch/2,
                f'{i}', ha='right', va='center', fontsize=7, color='#95a5a6')
ax_top.text(W_x0 - 0.08, W_y0_top + 0.06,
            'row', ha='right', va='bottom', fontsize=7, color='#95a5a6')

ax_top.text(W_x0 + 2*cw, W_y0_top + 0.28,
            'W   —   shape (8, 4)   —   8 rows × 4 columns',
            ha='center', va='bottom', fontsize=11, color=c_W, fontweight='bold')

# ── = operator ──────────────────────────────────────────────────────────────
eq_x = W_x0 + 4*cw + 0.28
ax_top.text(eq_x + 0.12, W_center_y, '=',
            ha='center', va='center', fontsize=24, color='#555', fontweight='bold')

# ── y: 1×4 HORIZONTAL row vector, right of =, centred on W ─────────────────
y_x0 = eq_x + 0.55
for j in range(4):
    fc = c_hl if j == 0 else c_y
    rect = plt.Rectangle((y_x0 + j*cw, row_y0), cw, ch,
                          fc=fc, ec='white', lw=1.5, alpha=0.88)
    ax_top.add_patch(rect)
    ax_top.text(y_x0 + j*cw + cw/2, row_y0 + ch/2,
                f'y[{j}]', ha='center', va='center',
                fontsize=7.5, color='white', fontweight='bold')

y_mid = y_x0 + 2*cw
ax_top.text(y_mid, row_y0 + ch + 0.20,
            'y   —   shape (4,)   —   result row vector',
            ha='center', va='bottom', fontsize=11, color=c_y, fontweight='bold')

# ── orange arc: col 0 of W → y[0] ──────────────────────────────────────────
col0_cx = W_x0 + cw/2
y0_cx   = y_x0 + cw/2

ax_top.annotate('', xy=(y0_cx, row_y0 + ch/2),
                xytext=(col0_cx, W_center_y - 1.0),
                arrowprops=dict(arrowstyle='->', color=c_hl, lw=2.0,
                                connectionstyle='arc3,rad=-0.30'))

# formula annotation (right-hand side)
ann_x = y_x0 + 4*cw + 0.4
ax_top.text(ann_x, row_y0 + ch + 0.15,
            'y[0] = x[0]·w[0,0] + x[1]·w[1,0] + … + x[7]·w[7,0]\n'
            '      = dot product of  x  with  col 0  of  W',
            ha='left', va='bottom', fontsize=9, color=c_hl, fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.35', fc='white', ec=c_hl, lw=1.3))

ax_top.text(ann_x, row_y0 - 0.20,
            'y[1] = x · col 1\ny[2] = x · col 2\ny[3] = x · col 3',
            ha='left', va='top', fontsize=9, color=c_y)

ax_top.text(ann_x, row_y0 - 1.55,
            'matmul = 4 dot products in parallel,\none per column of W',
            ha='left', va='top', fontsize=9.5, color='#2c3e50', fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.35', fc='#fdfefe', ec='#bdc3c7', lw=1))

# ── BOTTOM: dot product vs matmul ───────────────────────────────────────────
ax_bot.axis('off')
ax_bot.set_xlim(0, 14)
ax_bot.set_ylim(0, 2.2)

ax_bot.text(0.3, 2.1, 'Dot product  vs  Matrix multiply:',
            fontsize=11, fontweight='bold', color='#1c2833', va='top')
ax_bot.text(0.3, 1.70,
            '  u · v       two vectors of same length  →  one scalar          (8,) · (8,)  →  scalar',
            fontsize=9.5, color='#2c3e50', va='top')
ax_bot.text(0.3, 1.28,
            '  x @ W    row vector  ×  matrix           →  row vector         (8,) @ (8,4)  →  (4,)',
            fontsize=9.5, color='#2c3e50', va='top')
ax_bot.text(0.3, 0.86,
            '  matmul repeats the dot product once per column of W — output is a vector, not a scalar.',
            fontsize=9.5, color=c_hl, va='top', style='italic')

ax_bot.text(10.5, 1.95, '(8,) @ (8, 4)  →  (4,)',
            fontsize=12, fontweight='bold', color='#1c2833', va='top', ha='center',
            bbox=dict(boxstyle='round,pad=0.45', fc='#eaf2ff', ec='#1a5276', lw=1.5))
ax_bot.text(10.5, 1.18,
            'inner dims must match  (both 8)\nouter dims → output shape  (4,)',
            fontsize=9, color='#1a5276', va='top', ha='center', style='italic')

plt.tight_layout()
plt.show()

> **Matrix multiplication is not just a shape change.**
>
> `reshape` rearranges the *same* numbers into a new structure — no values change.
>
> `x @ W` computes *new* values: every output element is a weighted sum of **all** input elements.
> The weight matrix `W` is learned during training — it encodes which combinations of input features
> are useful for the task.
>
> | Operation | What changes | What stays the same |
> |-----------|-------------|-------------------|
> | `reshape` | structure (axes) | every value, `numel()` |
> | `x @ W`   | every value (mixed by W) | nothing — these are new numbers |
>
> In the Transformer, `x @ W` appears everywhere: Q/K/V projections, feedforward layers, output projection.
> None of those are about shape — they are about *what information to extract and recombine* from the input vector.

**Matrix multiplication computes many dot products at once.** Every entry in the output `A @ B` is the dot product between one row of `A` and one column of `B`. Consequently:

```python
scores = Q @ K.T          # shape (T, T)
# scores[i, j] == torch.dot(Q[i], K[j])
```

Every token's query vector is compared with every token's key vector in a single operation. You will see this exact computation in Chapter 7.

## 11. Why Vector Orientation Is a Convention

Mathematics often writes a vector as a column:

$$v = \begin{bmatrix} v_1 \\ v_2 \\ v_3 \end{bmatrix}$$

Code displays it as a row: `[v1, v2, v3]`.

Neither is "the true orientation." What matters is the **shape** and the **operation**.

This matters later when reading attention diagrams, matrix multiplication, and `QK^T`.

In [ ]:
v = torch.tensor([1.0, 2.0, 3.0])

print(f"Row vector (default): shape={v.shape}")
print(v)
print(f"\nColumn vector:        shape={v.unsqueeze(1).shape}  (same numbers, different layout)")
print(v.unsqueeze(1))

# Transpose is just a view — no data is copied
v_col = v.unsqueeze(1)   # (3,) → (3, 1)
v_row = v.unsqueeze(0)   # (3,) → (1, 3)
print(f"\nRow layout:    shape={v_row.shape}")
print(v_row)
print(f"\nv_col @ v_row -> outer product: {(v_col @ v_row).shape}")
print(v_col @ v_row)
print(f"\nv_row @ v_col -> dot product:   {(v_row @ v_col).shape}")
print(v_row @ v_col)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle('Row vector vs Column vector — same numbers, different shape convention',
             fontsize=12, fontweight='bold')

vals = [1.0, 2.0, 3.0]

# ── Panel 1: row vector (1, 3) ────────────────────────────────────────────────
ax = axes[0]
ax.axis('off'); ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.set_title('Row vector\nshape: (1, 3)', fontsize=10)
cw = 0.20; gap = 0.04; sx = 0.5 - len(vals)*(cw+gap)/2
for i, v in enumerate(vals):
    x = sx + i * (cw + gap)
    ax.add_patch(plt.Rectangle((x, 0.42), cw, 0.22, fc='#d6eaf8', ec='#1a5276', lw=1.5))
    ax.text(x + cw/2, 0.53, str(int(v)), ha='center', va='center',
            fontsize=13, color='#1c2833', fontweight='bold')
ax.text(0.5, 0.76, f'v.unsqueeze(0) → (1, 3)', ha='center', fontsize=9, color='#555')
ax.text(0.5, 0.24, 'arranged horizontally', ha='center', fontsize=9, color='#7f8c8d',
        style='italic')

# ── Panel 2: default 1-D vector ───────────────────────────────────────────────
ax = axes[1]
ax.axis('off'); ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.set_title('1-D tensor (default)\nshape: (3,)', fontsize=10)
for i, v in enumerate(vals):
    x = sx + i * (cw + gap)
    ax.add_patch(plt.Rectangle((x, 0.42), cw, 0.22, fc='#fdebd0', ec='#e67e22', lw=1.5))
    ax.text(x + cw/2, 0.53, str(int(v)), ha='center', va='center',
            fontsize=13, color='#1c2833', fontweight='bold')
ax.text(0.5, 0.76, 'torch.tensor([1, 2, 3])', ha='center', fontsize=9, color='#555')
ax.text(0.5, 0.24, 'no explicit orientation', ha='center', fontsize=9, color='#7f8c8d',
        style='italic')

# ── Panel 3: column vector (3, 1) ────────────────────────────────────────────
ax = axes[2]
ax.axis('off'); ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.set_title('Column vector\nshape: (3, 1)', fontsize=10)
ch2 = 0.18; gap2 = 0.04; sy2 = 0.5 - len(vals)*(ch2+gap2)/2
cw2 = 0.22
for i, v in enumerate(vals):
    y = sy2 + (len(vals) - 1 - i) * (ch2 + gap2)
    ax.add_patch(plt.Rectangle((0.5 - cw2/2, y), cw2, ch2,
                               fc='#d5f5e3', ec='#117a65', lw=1.5))
    ax.text(0.5, y + ch2/2, str(int(v)), ha='center', va='center',
            fontsize=13, color='#1c2833', fontweight='bold')
ax.text(0.5, 0.88, 'v.unsqueeze(1) → (3, 1)', ha='center', fontsize=9, color='#555')
ax.text(0.5, 0.10, 'arranged vertically', ha='center', fontsize=9, color='#7f8c8d',
        style='italic')

# Note box
for a in axes:
    a.add_patch(plt.Rectangle((0.02, 0.02), 0.96, 0.96, fc='none',
                               ec='#aab7b8', lw=1.0, ls='--'))

plt.tight_layout()
plt.show()

print("Key: shape (3,) vs (1,3) vs (3,1) — same 3 values, different operation behaviour.")
print("  v_col @ v_row  →  outer product  (3,1) @ (1,3) → (3,3)")
print("  v_row @ v_col  →  dot product    (1,3) @ (3,1) → (1,1)")

## 12. Reshaping and Flattening

### Can the whole sequence be treated as one vector?

> **This is the central question of the first half of this book.** How you answer it determines the architecture of your model.

Yes — a sequence of shape `(T, C)` can be flattened into one `(T*C)`-dimensional vector.

```python
z_flat = z.reshape(B, T * C)
```

But *should* you? And if so, what does the model actually gain? Let us trace the journey.

---

**Where we started: the bigram.**

In Chapter 1, every prediction used exactly one token of context. The model sees only the current token when predicting the next one. It therefore has one-token context and no mechanism for combining information across multiple preceding positions.

---

**What flattening gives you.**

If we flatten the entire sequence from `(T, C)` to `(T*C,)` and feed that to a feedforward layer, the model receives all T token vectors at once. Position 3 can influence the prediction at position 7. **Cross-token context is now possible.**

This is the idea behind a **fixed-context window model**, which Chapter 6 builds.

---

**The cost: dimension explodes.**

| T | C | Flattened dimension |
|---|---|---------------------|
| 8 | 16 | 128 |
| 128 | 768 | 98,304 |
| 1,000 | 768 | 768,000 |
| 8,192 | 4,096 | 33,554,432 |

A feedforward layer receiving a 33M-dimensional input is not practical.

---

**The implicit position problem.**

A fixed-window MLP actually does know position — structurally. Token 0's values always occupy indices `[0:C]`, token 1's always occupy `[C:2C]`, and so on. Position is encoded in which coordinate range each token inhabits.

The real problem is that position becomes **rigid**. A weight useful at position 3 cannot be reused at position 7. Increase the context window and the model needs an entirely new set of weights.

**Self-attention does not share this rigidity — but it introduces a different requirement.** Attention applies the same projections at every position, which avoids rigid parameter growth. But self-attention alone has no inherent notion of absolute or relative order — if the token vectors are permuted, the resulting representations are permuted correspondingly. It cannot distinguish position 2 from position 6 without an explicit signal. Self-attention reuses the same learned transformations across positions, so sequence order must be supplied by a separate positional mechanism. We will introduce that mechanism in Chapter 7.

---

**The roadmap.**

| Model | What it sees | Cross-token context? |
|-------|-------------|----------------------|
| Bigram (Ch 1) | One token at a time | No |
| Fixed-context window (Ch 6) | All T tokens, flattened to `(T*C,)` | Yes — but rigid and expensive |
| Self-attention (Ch 7) | All T tokens, shape stays `(T, C)` | Yes — without flattening |

Every architecture in the second half of this book is an answer to the question this section asked.


In [ ]:
# Flatten a sequence into one large vector
B, T, C = 4, 8, 16
z = torch.randn(B, T, C)

z_flat = z.reshape(B, T * C)
print(f'Original:  {z.shape}')
print(f'Flattened: {z_flat.shape}')
print(f"\nFlattened dimension = T x C = {T} x {C} = {T * C}")

print("\nHow flattened dimension scales with context length:")
for t, c in [(8, 16), (128, 768), (1000, 768), (8192, 4096)]:
    print(f'  T={t:5d}, C={c:5d} -> {t * c:,} dimensions')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Reshaping and Flattening', fontsize=13, fontweight='bold')

# ── Panel 1: flattened dim growth bar chart ───────────────────────────────────
ax = axes[0]
cases     = [(8, 16), (128, 768), (1000, 768), (8192, 4096)]
labels    = [f'T={t}, C={c}' for t, c in cases]
flat_dims = [t * c for t, c in cases]
bar_colors = ['#3498db', '#2ecc71', '#e67e22', '#e74c3c']

ax.barh(range(len(cases)), flat_dims, color=bar_colors, edgecolor='white', height=0.55)
for i, (fd, clr) in enumerate(zip(flat_dims, bar_colors)):
    ax.text(fd + flat_dims[-1]*0.01, i, f'{fd:,}',
            va='center', fontsize=9.5, fontweight='bold', color=clr)

ax.set_yticks(range(len(cases)))
ax.set_yticklabels(labels, fontsize=9.5)
ax.set_xlabel('Flattened dimension  T × C', fontsize=10)
ax.set_title('Flattened dim grows with context length\n(B, T, C) → (B, T×C)', fontsize=10)
ax.set_xscale('log')
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.grid(axis='x', alpha=0.3)
ax.spines[['top', 'right']].set_visible(False)
ax.axvline(128*768, color='#7f8c8d', lw=1.2, ls='--', alpha=0.6)
ax.text(128*768+3000, 3.55, 'GPT-2 small\n(T=1024, C=768)',
        fontsize=7.5, color='#7f8c8d', va='top')

# ── Panel 2: 4×6 matrix → flat vector ───────────────────────────────────────
ax2 = axes[1]
ax2.axis('off')
ax2.set_xlim(0, 6)
ax2.set_ylim(0, 5)
ax2.set_title('Reshape changes structure, not element count', fontsize=10)

T_v, C_v = 4, 6
cw, ch, gap = 0.75, 0.5, 0.06

start_x, start_y = 0.3, 2.0

# (T, C) matrix
for t in range(T_v):
    for c in range(C_v):
        x = start_x + c * (cw + gap)
        y = start_y + (T_v - 1 - t) * (ch + gap)
        ax2.add_patch(plt.Rectangle((x, y), cw, ch,
                                    fc='#aed6f1', ec='#1a5276', lw=1.0))
        ax2.text(x + cw/2, y + ch/2, f'{t*C_v+c}',
                 ha='center', va='center', fontsize=7.5, color='#1c2833')

matrix_width = C_v*(cw+gap) - gap   # exact pixel-width of the matrix above

ax2.text(start_x + matrix_width/2, start_y - 0.28,
         f'shape: ({T_v}, {C_v})   ←  T rows × C cols',
         ha='center', fontsize=9, color='#1a5276', fontweight='bold')

# reshape arrow
mid_y = start_y + T_v*(ch+gap)/2 + 0.1
ax2.annotate('', xy=(0.25, mid_y - 0.7), xytext=(0.25, mid_y - 0.1),
             arrowprops=dict(arrowstyle='->', color='#e74c3c', lw=2.0))
ax2.text(0.28, mid_y - 0.4, '.reshape\n(-1)', fontsize=8.5,
         color='#e74c3c', fontweight='bold', ha='left')

# flat vector — 24 cells spanning the SAME total width as the matrix
flat_y = 1.0
gap_f  = 0.02                                              # tighter gap for flat cells
fw     = (matrix_width - (T_v*C_v - 1)*gap_f) / (T_v*C_v)  # ≈ 0.18

for k in range(T_v * C_v):
    x = start_x + k * (fw + gap_f)
    ax2.add_patch(plt.Rectangle((x, flat_y), fw, ch*0.85,
                                fc='#fadbd8', ec='#922b21', lw=1.0))
    ax2.text(x + fw/2, flat_y + ch*0.85/2, f'{k}',
             ha='center', va='center', fontsize=6, color='#7b241c')

ax2.text(start_x + matrix_width/2, flat_y - 0.28,
         f'shape: ({T_v*C_v},)   ←  T×C = {T_v}×{C_v} elements',
         ha='center', fontsize=9, color='#922b21', fontweight='bold')

# numel callout
ax2.text(start_x + matrix_width/2, 4.6,
         f'numel before = numel after = {T_v*C_v}',
         ha='center', fontsize=10, color='#117a65', fontweight='bold',
         bbox=dict(boxstyle='round,pad=0.4', fc='#d5f5e3', ec='#117a65', lw=1.5))

plt.tight_layout()
plt.show()

## 13. High-Dimensional Spaces

Modern models work in hundreds or thousands of dimensions — and they never operate on just two vectors. A batch of T tokens gives T vectors simultaneously, each C-dimensional. Every operation we built in 2D and 3D extends without modification.

Four things remain true at any dimension:

- **Dot product, norm, cosine similarity** — same formulas, same meaning
- **Addition** — still a vector in the same space
- **Matmul** — the row-by-column rule does not care how large rows or columns are
- **Distance and similarity between composites** — a composite vector is just a vector

The only thing we lose is the ability to draw the space.

> **Near-zero cosine, not near-zero dot product.** For random high-dimensional vectors, the **cosine similarity** tends toward zero — directions are approximately orthogonal. The raw dot product itself can easily be ±10 or ±40 in absolute terms; it is the magnitude-normalised version that vanishes. Your notebook makes this visible: the pairwise heatmap shows raw dot products on the diagonal (large, equal to ‖·‖²) and off-diagonal entries that vary but whose cosine similarities are near zero.

The geometry that develops in a learned representation space depends on how the model is trained and on the data it learns from. Understanding what that geometry looks like — and whether it carries useful structure — is one of the questions Chapter 3 investigates through experiment.


In [ ]:
# Same operations — four vectors, high dimensions
d = 512

torch.manual_seed(0)
u = torch.randn(d)
v = torch.randn(d)
w = torch.randn(d)
x = torch.randn(d)

vecs  = [u, v, w, x]
names = ['u', 'v', 'w', 'x']

# ── 1. Pairwise dot products ───────────────────────────────────────────────────
print("1. Pairwise dot products (upper triangle)")
for i, (a, na) in enumerate(zip(vecs, names)):
    for j, (b, nb) in enumerate(zip(vecs, names)):
        if j >= i:
            print(f"   {na}·{nb} = {torch.dot(a, b).item():>9.2f}", end="   ")
    print()

# ── 2. Vector addition ────────────────────────────────────────────────────────
print("\n2. Addition — cumulative norms")
running = torch.zeros(d)
for a, na in zip(vecs, names):
    running = running + a
    print(f"   +{na}  ->  ||cumulative|| = {running.norm():.2f}")

# ── 3. Matmul — stack as rows, chain transforms ───────────────────────────────
print("\n3. Matmul — U = stack([u,v,w,x]) -> (4,d) @ (d,64) -> (4,64) @ (64,8) -> (4,8)")
U  = torch.stack([u, v, w, x])   # (4, d) — four row vectors
W1 = torch.randn(d, 64)          # (d, 64) first transform
W2 = torch.randn(64, 8)          # (64, 8) second transform
Y1 = U @ W1                      # (4, 64)
Y2 = Y1 @ W2                     # (4, 8)
print(f"   U  shape: {U.shape}")
print(f"   Y1 = U @ W1   shape: {Y1.shape}")
print(f"   Y2 = Y1 @ W2  shape: {Y2.shape}   <- row/col rule holds at every step")

# ── 4. Distance between composites ────────────────────────────────────────────
print("\n4. Distance")
print(f"   ||u - w||             = {(u - w).norm():.2f}")
print(f"   ||(u+v) - w||         = {((u + v) - w).norm():.2f}")
print(f"   ||(u+v) - (w+x)||     = {((u + v) - (w + x)).norm():.2f}")
print(f"   ||u - v + w - x||     = {(u - v + w - x).norm():.2f}")

# ── 5. Cosine similarity between composites ───────────────────────────────────
def cos(a, b):
    return F.cosine_similarity(a.unsqueeze(0), b.unsqueeze(0)).item()

print("\n5. Cosine similarity")
print(f"   sim(u, v)          = {cos(u, v):.4f}")
print(f"   sim(u, w)          = {cos(u, w):.4f}")
print(f"   sim(u+v, w)        = {cos(u + v, w):.4f}")
print(f"   sim(u+v, w+x)      = {cos(u + v, w + x):.4f}")
print(f"   sim(u+v, u+v)      = {cos(u + v, u + v):.4f}   <- same vector -> 1")


In [ ]:
import numpy as np

d = 512
torch.manual_seed(0)
u = torch.randn(d); v = torch.randn(d)
w = torch.randn(d); x = torch.randn(d)

vecs  = [u, v, w, x]
names = ['u', 'v', 'w', 'x']

def cos(a, b):
    return F.cosine_similarity(a.unsqueeze(0), b.unsqueeze(0)).item()

fig, axes = plt.subplots(1, 3, figsize=(15, 4.8))
fig.suptitle('High-Dimensional Spaces — four vectors, d = 512', fontsize=12, fontweight='bold')

# ── Panel 1: pairwise dot product heatmap ────────────────────────────────────
ax = axes[0]
dot_mat = np.array([[torch.dot(a, b).item() for b in vecs] for a in vecs])
vmax = float(np.abs(dot_mat).max())
im = ax.imshow(dot_mat, cmap='coolwarm', aspect='auto', vmin=-vmax, vmax=vmax)
ax.set_xticks(range(4)); ax.set_xticklabels(names)
ax.set_yticks(range(4)); ax.set_yticklabels(names)
ax.set_title('Pairwise dot products\n(diagonal = ||·||²)', fontsize=10)
for i in range(4):
    for j in range(4):
        val = dot_mat[i, j]
        color = 'white' if abs(val) > 0.55 * vmax else '#1c2833'
        ax.text(j, i, f'{val:.0f}', ha='center', va='center', fontsize=9, color=color)
plt.colorbar(im, ax=ax, shrink=0.8)

# ── Panel 2: distances between composite vectors ──────────────────────────────
ax2 = axes[1]
dist_cases = [
    ('||u - w||',          (u - w).norm().item()),
    ('||(u+v) - w||',      ((u + v) - w).norm().item()),
    ('||(u+v)\n-(w+x)||',  ((u + v) - (w + x)).norm().item()),
    ('||u-v+w-x||',        (u - v + w - x).norm().item()),
]
d_labels = [c[0] for c in dist_cases]
d_vals   = [c[1] for c in dist_cases]
colors_d = ['#2e86c1', '#27ae60', '#e67e22', '#8e44ad']
bars2 = ax2.bar(range(len(dist_cases)), d_vals, color=colors_d, alpha=0.85, edgecolor='white', width=0.55)
ax2.set_xticks(range(len(dist_cases)))
ax2.set_xticklabels(d_labels, fontsize=8.5)
ax2.set_title('Euclidean distance — composite vectors', fontsize=10)
ax2.set_ylabel('‖ · ‖')
for bar, val in zip(bars2, d_vals):
    ax2.text(bar.get_x() + bar.get_width() / 2, val + 0.4, f'{val:.1f}',
             ha='center', va='bottom', fontsize=9)
ax2.set_ylim(0, max(d_vals) * 1.18)

# ── Panel 3: cosine similarity — composites ───────────────────────────────────
ax3 = axes[2]
sim_cases = [
    ('sim(u, v)',       cos(u, v)),
    ('sim(u, w)',       cos(u, w)),
    ('sim(u+v, w)',     cos(u + v, w)),
    ('sim(u+v, w+x)',   cos(u + v, w + x)),
    ('sim(u+v, u+v)',   cos(u + v, u + v)),
]
s_labels = [c[0] for c in sim_cases]
s_vals   = [c[1] for c in sim_cases]
colors_s = ['#2e86c1', '#27ae60', '#e67e22', '#c0392b', '#1abc9c']
y_pos = range(len(sim_cases))
bars3 = ax3.barh(y_pos, s_vals, color=colors_s, alpha=0.85, edgecolor='white', height=0.55)
ax3.set_yticks(y_pos)
ax3.set_yticklabels(s_labels, fontsize=9)
ax3.axvline(0, color='gray', lw=0.7)
ax3.set_xlim(-0.25, 1.25)
ax3.set_title('Cosine similarity — composite vectors', fontsize=10)
ax3.set_xlabel('cosine similarity')
for bar, val in zip(bars3, s_vals):
    xp = val + 0.02 if val >= 0 else val - 0.02
    ha = 'left' if val >= 0 else 'right'
    ax3.text(xp, bar.get_y() + bar.get_height() / 2,
             f'{val:.3f}', va='center', ha=ha, fontsize=8.5)

plt.tight_layout()
plt.show()


## 14. PyTorch Lab — Becoming Fluent in Vector Shapes

Eleven exercises using the tinyshakespeare vocabulary (95 chars, same as Chapter 1).

**Why 95 numbers per position?** The feature dimension `C` determines how many numerical coordinates each position has:

```text
C =   2  → two numerical coordinates per token
C =  64  → 64 numerical coordinates per token
C =  768  → GPT-2 small uses 768 coordinates per token
C = 4096  → Llama 3 8B uses a hidden size of 4096
```

The model may distribute useful information across many coordinates simultaneously — one coordinate does not equal one feature. This is *distributed representation*.

In Chapter 1, `C = vocab_size = 95`; each selected row contained 95 logits used directly for next-character prediction. Chapter 3 will separate `C` from `vocab_size` and explore what that separation makes possible.


In [ ]:
# Exercise 1 — Inspect vector properties
v = torch.tensor([1.0, 2.0, 3.0])

print(f"v        = {v.tolist()}")
print(f"v.shape  = {v.shape}    ← (3,) means 1 axis with 3 entries")
print(f"v.ndim   = {v.ndim}       ← rank 1 (one axis)")
print(f"v.numel()= {v.numel()}       ← 3 stored values")
print(f"v.dtype  = {v.dtype}  ← 32-bit floating-point")

ids = torch.tensor([10, 20, 30])
print(f"\nids.dtype = {ids.dtype}  ← integer (for token IDs)")

In [ ]:
# Exercise 2 — Norm
v = torch.tensor([3.0, 4.0])
norm_v = torch.linalg.vector_norm(v)
print(f"v = {v.tolist()}")
print(f"||v|| = {norm_v:.4f}  ← predict this before running: sqrt(3² + 4²) = ?")

In [ ]:
# Exercise 3 — Vector addition and scalar multiplication
u = torch.tensor([1.0, 2.0, 3.0])
v = torch.tensor([4.0, 5.0, 6.0])

print(f"u       = {u.tolist()}")
print(f"v       = {v.tolist()}")
print(f"u + v   = {(u + v).tolist()}")
print(f"0.5 * v = {(0.5 * v).tolist()}")

# Later connections
x   = torch.tensor([0.5, -0.3, 0.8])
F_x = torch.tensor([0.1,  0.2, -0.1])
print(f"\nResidual connection preview: x + F(x) = {(x + F_x).tolist()}")
print("  → same operation; the mechanism that uses it arrives in Chapter 10.")

In [ ]:
# Exercise 4 — Dot product (calculate manually first, then verify)
u = torch.tensor([1.0, 2.0, 3.0])
v = torch.tensor([4.0, 5.0, 6.0])

# Manual: 1×4 + 2×5 + 3×6 = ?
manual = 1*4 + 2*5 + 3*6
print(f"Manual: 1×4 + 2×5 + 3×6 = {manual}")

dot = torch.dot(u, v)
print(f"torch.dot(u, v) = {dot.item():.2f}  ← matches?")

In [ ]:
# Exercise 5 — Euclidean distance
u = torch.tensor([1.0, 2.0, 3.0])
v = torch.tensor([4.0, 5.0, 6.0])

dist = torch.linalg.vector_norm(u - v)
print(f"u = {u.tolist()}")
print(f"v = {v.tolist()}")
print(f"||u - v|| = {dist:.4f}  ← how far apart in space")

In [ ]:
# Exercise 6 — Cosine similarity: observe what changes when you scale a vector
u = torch.tensor([1.0, 2.0, 3.0])
v = torch.tensor([4.0, 5.0, 6.0])
v_scaled = 10.0 * v   # same direction, 10× larger

cos_uv        = F.cosine_similarity(u.unsqueeze(0), v.unsqueeze(0)).item()
cos_uv_scaled = F.cosine_similarity(u.unsqueeze(0), v_scaled.unsqueeze(0)).item()
dot_uv        = torch.dot(u, v).item()
dot_uv_scaled = torch.dot(u, v_scaled).item()
dist_uv       = torch.linalg.vector_norm(u - v).item()
dist_uv_scaled= torch.linalg.vector_norm(u - v_scaled).item()

print(f"Original v:      cos={cos_uv:.4f}  dot={dot_uv:.2f}  dist={dist_uv:.4f}")
print(f"v scaled by 10:  cos={cos_uv_scaled:.4f}  dot={dot_uv_scaled:.2f}  dist={dist_uv_scaled:.4f}")
print()
print("Observation:")
print("  cosine similarity unchanged — it compares direction only")
print("  dot product scales up       — it is sensitive to magnitude")
print("  Euclidean distance scales up — it is also sensitive to magnitude")

In [ ]:
# Exercise 7 — Matrix multiplication: predict the output shape before running
x = torch.randn(3)
W = torch.randn(3, 5)

# Before running: what will y.shape be?
# Trace: (3,) @ (3, 5) → ?
y = x @ W

print(f"x shape: {x.shape}")
print(f"W shape: {W.shape}")
print(f"y shape: {y.shape}")
print(f"\nTrace: ({x.shape[0]},) @ ({W.shape[0]}, {W.shape[1]}) → ({y.shape[0]},)")
print(f"\nA {x.shape[0]}-dimensional vector was transformed into a {y.shape[0]}-dimensional vector.")

In [ ]:
# Exercise 8 — Revisit a Chapter 1 batch
xb, yb = get_batch('train')   # shape: (batch_size, block_size) = (32, 8)

print(f"xb.shape  = {xb.shape}")
print(f"  axis 0  = batch   ({xb.shape[0]} sequences)")
print(f"  axis 1  = context ({xb.shape[1]} positions)")
print(f"  ndim    = {xb.ndim}     ← rank-2 tensor")
print(f"  dtype   = {xb.dtype}  ← integer token IDs, not floating-point vectors")
print(f"  numel() = {xb.numel()} ← total token ID values stored")
print(f"\nSample row: {xb[0].tolist()}")
print(f"\nThese are token IDs — discrete indices into the vocabulary.")
print(f"They are NOT yet learned representation vectors. That transformation arrives in Chapter 3.")

In [ ]:
# Exercise 9 — Preview (B, T, C) slicing (shape practice — random tensor, no learned values)
B, T, C = 4, 8, 16
z = torch.randn(B, T, C)   # random values for shape practice only

print(f"z.shape     = {z.shape}   ← (B, T, C)")
print(f"z[0].shape  = {z[0].shape}     ← one sequence: T positions × C features")
print(f"z[0,0].shape= {z[0,0].shape}      ← one position vector: C-dimensional")
print(f"z[0,:,0].shape={z[0,:,0].shape}     ← one feature traced across T positions")
print()
print("Interpretation of each slice:")
print("  z[b, :, :] → one T×C sequence matrix")
print("  z[b, t, :] → one C-dimensional position vector")
print("  z[b, :, c] → one feature signal across positions")
print()
print("Note: these are random numbers. Chapter 3 replaces them with")
print("actual trainable token representations learned through optimization.")

In [ ]:
# Exercise 10 — Flatten one sequence
B_actual = xb.shape[0]
T_actual = xb.shape[1]
C_flat   = 16   # hypothetical feature dimension for illustration

z = torch.randn(B_actual, T_actual, C_flat)
z_flat = z.reshape(B_actual, T_actual * C_flat)

print(f"Original:  {z.shape}")
print(f"Flattened: {z_flat.shape}")
print(f"numel unchanged: {z.numel() == z_flat.numel()}")
print()
print("What changed:  the structural axes — T and C are merged into one axis.")
print("What did not:  the number of values. reshape moves data, not deletes it.")
print()
print("Later we will build a model that uses exactly this concatenation as its")
print("context representation. That experiment (Chapter 6) will make concrete")
print("why fixed concatenation becomes limiting as context grows.")

In [ ]:
# Exercise 11 — Q @ Kᵀ pairwise scores
# Before checking the assertion: manually compute scores[0, 1] as dot(Q[0], K[1])
Q = torch.tensor([[1., 2., 3.], [4., 5., 6.]])
K = torch.tensor([[7., 8., 9.], [1., 0., 1.]])
scores = Q @ K.T
print("scores shape:", scores.shape)   # (2, 2)
print("scores:")
print(scores)
# Verify: every entry is a dot product
assert torch.isclose(scores[0, 1], torch.dot(Q[0], K[1]))
print("Verified: scores[0, 1] ==", scores[0, 1].item())
print("  = dot([1,2,3], [1,0,1]) = 1*1 + 2*0 + 3*1 = 4")
# This is the pairwise attention score computation from Chapter 7.

## 15. Research Connections: How Papers Enter This Chapter

Chapter 2 does not contain a conventional related-work section. Papers arrive when the reader has built the primitive needed to understand why the paper matters.

### After the dot product

**Vaswani et al. (2017), *Attention Is All You Need***

> The Transformer compares query and key vectors using dot products. The operation is simple — multiply coordinates, sum the products. The self-attention chapter will build the full mechanism; Chapter 2 supplies the primitive it rests on.

### After cosine similarity and vector geometry

**Mikolov et al. (2013), *Efficient Estimation of Word Representations in Vector Space***

> Trained vector spaces can develop systematic geometric relationships — relationships measurable through the cosine similarity and distance tools introduced in this chapter. Chapter 3 will investigate how characters acquire learned vector representations, then measure what geometry emerges after training.

### At the transition to Chapter 3

**Bengio et al. (2003), *A Neural Probabilistic Language Model***

Chapter 1's bigram table mapped a discrete character directly to prediction logits. A different possibility is to first map a discrete symbol into a learned distributed representation and let a separate computation turn that representation into a prediction. Neural language modeling work such as Bengio et al. made this idea central long before modern Transformers.

> If the input is an integer ID, how does the model learn that internal vector?

That question opens Chapter 3.

---

**Rule for the whole book:** Do not introduce a paper because it is famous. Introduce it when the reader can answer:
1. What problem was present?
2. What primitive do I already understand?
3. What did the paper do with that primitive?
4. Which parts must wait because I have not built them yet?

## 16. Where These Vector Operations Reappear

```text
vector addition
    ↓
residual connections (Chapter 10)
```

```text
dot product
    ↓
attention — query · key comparison (Chapter 7)
```

```text
matrix multiplication
    ↓
neural projections, feedforward layers, Q/K/V, LoRA (Chapters 7–15)
```

```text
distance / cosine similarity
    ↓
representation analysis, retrieval, RAG (Chapter 3 onward)
```

```text
(B, T, C)
    ↓
hidden representations throughout every Transformer chapter
```

The goal is not to teach these systems yet. It is to show that Chapter 2's mathematics will be reused rather than discarded.

## 17. What Chapter 2 Deliberately Does Not Explain

At the end of Chapter 2 the following questions should remain unresolved:

- Where do the `C` values associated with a token come from?
- Why should different tokens acquire different vectors?
- How does training change those vectors?
- Why can learned vectors develop useful geometry?
- Why can neighboring vectors become meaningful?
- How does one compact representation eventually produce vocabulary logits?
- How does context later change a token's representation?

These are not gaps in Chapter 2. They are the reason Chapter 3 exists.

---

## Chapter Summary

- A scalar is one number. A vector is an ordered collection of coordinates. A matrix is a rank-2 tensor viewable as rows, columns, or a transformation. A tensor may have still more axes.
- **Rank** = number of axes (`.ndim`). **Shape** = size of each axis. **`numel()`** = total stored values. **`dtype`** = numerical type.
- `(B, T)` is a batch of integer token-ID sequences. `(B, T, C)` gives `C` numerical values at every position.
- `z[b, t, :]` is one `C`-dimensional vector. `z[b, :, :]` is one `T×C` sequence matrix. `z[b, :, c]` is one feature traced across positions.
- Dot products, norms, distances, and cosine similarity give us ways to measure vectors.
- Matrix multiplication transforms one vector space into another: `(in,) @ (in, out) → (out,)`.
- Reshaping changes organisation, not the number of values.
- High-dimensional spaces obey the same operations even when we cannot draw them.
- These primitives will reappear in learned representations, attention, neural layers, inference, and model adaptation.

---

## What's Next

Chapter 2 has taught us the mathematical object. We can read `(B, T, C)` and understand exactly what it means.

But one question was left intentionally unanswered:

> **When the input is only a token ID, where do those `C` useful numbers come from?**

That leads directly to **Chapter 3 — Embeddings: From IDs to Learned Representations**.